# 01 · Exploratory data analysis

Everything below is computed from `data/raw/` and `data/processed/` when the notebook runs. Observations are printed by code, not typed by hand.

Re-run: `python scripts/prepare_dataset.py && python scripts/inspect_dataset.py && jupyter nbconvert --to notebook --execute --inplace notebooks/01_eda.ipynb`

In [1]:
import json, sys, collections
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display
from src.config import FIGURES, METRICS, URGENCY_LABELS
from src.data import load_raw, load_split
from src.preprocessing import detect_script
FIGURES.mkdir(parents=True, exist_ok=True)
rows = [dict(r, split=s) for s in ('train','val','test') for r in load_split(s)]
df = pd.DataFrame(rows)
df['source_kind'] = np.where(df.intent.notna(), 'Banking77 (intent)', 'synthetic_enterprise (urgency/escalation)')
df['script'] = df.text.map(detect_script)
audit = json.loads((METRICS/'data_audit.json').read_text())
print(df.shape)

(8038, 12)


## 1. Raw subsets inspected (provenance fields as shipped in the files)

In [2]:
raw = pd.DataFrame({k: {'items': v['n'], 'origin': ', '.join(v['origins']), 'license': ', '.join(v['licenses']),
        'languages': len(v['languages']), 'families': ', '.join(v['families']), 'dup ids': v['duplicate_ids'],
        'missing fields': sum(v['missing_fields'].values())} for k, v in audit['raw'].items()}).T
display(raw)
for k, v in audit['raw'].items(): print(k, '->', v['provenance_notes'])

,items,origin,license,languages,families,dup ids,missing fields
fintech_banking77,7816,"banking77, banking77+mt",CC BY 4.0,7,intent,0,0
synthetic_enterprise,444,synthetic,CC BY 4.0,2,"urgency, escalation, routing",0,0
hinglish_lid,3220,comilingua,CC BY 4.0,1,lid,0,0
intent_massive,57922,massive,CC BY 4.0,7,intent,0,0


fintech_banking77 -> ['Banking77 translated to Indic via NLLB-200']
synthetic_enterprise -> ['Synthetic enterprise data generated by cmul8 (qwen3-235b + deepseek checker)']
hinglish_lid -> ['COMI-LINGUA source, reshaped in v1 to script-detection MCQ']
intent_massive -> ['MASSIVE official test split']


## 2. Processed dataset size by split and source

In [3]:
display(pd.crosstab(df.split, df.source_kind, margins=True))
print('Labelled rows per task per split:')
display(df.groupby('split').agg(intent=('intent', lambda s: s.notna().sum()), urgency=('urgency', lambda s: s.notna().sum()), escalation=('escalation', lambda s: s.notna().sum())))
overlap = (df.intent.notna() & (df.urgency.notna() | df.escalation.notna())).sum()
print('Rows carrying BOTH an intent label and an urgency/escalation label:', int(overlap))

source_kind,Banking77 (intent),synthetic_enterprise (urgency/escalation),All
split,,,
test,1173,34,1207
train,5471,155,5626
val,1172,33,1205
All,7816,222,8038


Labelled rows per task per split:


,intent,urgency,escalation
split,,,
test,1173,27,26
train,5471,124,123
val,1172,24,28


Rows carrying BOTH an intent label and an urgency/escalation label: 0


## 3. Intent distribution (Banking77, 77 classes)

In [4]:
ic = df[df.intent.notna()].groupby('intent').size().sort_values()
fig, ax = plt.subplots(figsize=(7, 14), dpi=100)
ax.barh(ic.index, ic.values, color='#2459d6'); ax.tick_params(axis='y', labelsize=6)
ax.set_xlabel('rows (all splits)'); ax.set_title(f'Intent distribution ({len(ic)} classes)'); fig.tight_layout()
fig.savefig(FIGURES/'intent_distribution.png'); plt.show()
print('classes:', len(ic), '| min:', ic.min(), ic.idxmin(), '| max:', ic.max(), ic.idxmax(), f'| max/min = {ic.max()/ic.min():.2f}')

classes: 77 | min: 49 contactless_not_working | max: 150 transaction_charged_twice | max/min = 3.06


## 4. Urgency distribution (synthetic_enterprise only)

In [5]:
u = df[df.urgency.notna()].copy(); u['level'] = u.urgency.astype(int).map(lambda i: URGENCY_LABELS[i])
tab = pd.crosstab(u.level, u.lang).reindex(URGENCY_LABELS).fillna(0).astype(int)
fig, ax = plt.subplots(figsize=(6, 3.8), dpi=110); tab.plot.bar(ax=ax, rot=0); ax.set_ylabel('texts'); ax.set_title('Urgency distribution (synthetic, n=%d)' % len(u)); fig.tight_layout()
fig.savefig(FIGURES/'urgency_distribution.png'); plt.show(); display(tab)

lang,hi-Deva,hi-Latn
level,,
very_low,0,1
low,7,5
medium,15,22
high,27,30
critical,38,30


## 5. Escalation distribution (synthetic_enterprise only)

In [6]:
e = df[df.escalation.notna()].copy(); e['label'] = e.escalation.astype(int).map({0: 'no', 1: 'yes'})
tab = pd.crosstab(e.label, e.lang)
fig, ax = plt.subplots(figsize=(5, 3.6), dpi=110); tab.plot.bar(ax=ax, rot=0); ax.set_ylabel('texts'); ax.set_title('Escalation (human needed?) (synthetic, n=%d)' % len(e)); fig.tight_layout()
fig.savefig(FIGURES/'escalation_distribution.png'); plt.show(); display(tab)
print(f'positive rate: {e.escalation.mean():.3f}')

lang,hi-Deva,hi-Latn
label,,
no,25,33
yes,62,57


positive rate: 0.672


## 6. Language / script distribution

In [7]:
tab = pd.crosstab(df.lang, df.split)[['train','val','test']]
fig, ax = plt.subplots(figsize=(7, 3.8), dpi=110); tab.plot.bar(ax=ax, rot=30, stacked=True); ax.set_ylabel('rows'); ax.set_title('Rows by language tag and split'); fig.tight_layout()
fig.savefig(FIGURES/'language_distribution.png'); plt.show(); display(tab)
display(pd.crosstab(df.lang, df.script))

split,train,val,test
lang,,,
bn-Beng,749,159,167
en-Latn,908,211,199
hi-Deva,835,166,176
hi-Latn,94,20,21
kn-Knda,780,150,164
ml-Mlym,770,169,140
ta-Taml,753,146,165
te-Telu,737,184,175


script,Beng,Deva,Deva+Latn,Knda,Latn,Mlym,Taml,Telu,mixed
lang,,,,,,,,,
bn-Beng,1051,0,0,0,0,0,0,0,24
en-Latn,0,0,0,0,1318,0,0,0,0
hi-Deva,0,1138,39,0,0,0,0,0,0
hi-Latn,0,0,0,0,135,0,0,0,0
kn-Knda,0,0,0,1082,0,0,0,0,12
ml-Mlym,0,0,0,0,0,1067,0,0,12
ta-Taml,0,0,0,0,0,0,994,0,70
te-Telu,0,0,0,0,0,0,0,1023,73


## 7. Text length

In [8]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained('google/muril-base-cased')
df['n_chars'] = df.text.str.len(); df['n_tokens'] = [len(x) for x in tok(df.text.tolist())['input_ids']]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), dpi=110)
for name, g in df.groupby('source_kind'):
    axes[0].hist(g.n_chars, bins=50, alpha=.6, label=name.split(' ')[0]); axes[1].hist(g.n_tokens, bins=50, alpha=.6, label=name.split(' ')[0])
axes[0].set_xlabel('characters'); axes[1].set_xlabel('MuRIL WordPiece tokens (incl. [CLS]/[SEP])'); axes[0].legend(); fig.suptitle('Text length distribution'); fig.tight_layout()
fig.savefig(FIGURES/'text_length_distribution.png'); plt.show()
display(df.groupby('lang')[['n_chars','n_tokens']].agg(['mean','median','max']).round(1))
q = np.percentile(df.n_tokens, [50, 90, 95, 99, 100]); print('token percentiles (50/90/95/99/100):', q, '| >64 tokens:', int((df.n_tokens > 64).sum()), 'of', len(df))

/usr/local/lib/python3.13/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


n_chars             n_tokens            
           mean median  max     mean median  max
lang                                            
bn-Beng    60.8   49.0  373     15.3   13.0   85
en-Latn    58.2   47.0  342     15.9   13.0   76
hi-Deva    67.4   56.0  333     18.2   15.0   80
hi-Latn    67.5   64.0  123     18.4   17.0   32
kn-Knda    66.0   52.0  369     16.6   14.0   81
ml-Mlym    72.4   57.0  492     18.3   15.0  112
ta-Taml    72.1   58.0  404     15.5   13.0   77
te-Telu    61.5   49.0  310     18.2   15.0   87

token percentiles (50/90/95/99/100): [ 14.  28.  37.  56. 112.] | >64 tokens: 40 of 8038


## 8. Quality: missing / malformed / duplicates / leakage

In [9]:
print('missing text:', int(df.text.isna().sum()), '| empty text:', int((df.text.str.strip() == '').sum()))
print('Exact (lang,text) overlap between splits:', audit['leakage']['exact_text_overlap'])
print('ID overlap between splits:', audit['leakage']['id_overlap'])
print('Near duplicates test->train (char 3-5gram TF-IDF cosine >= 0.85):')
print(json.dumps(audit['leakage']['near_duplicates_test_vs_train'], indent=1))
print(audit['leakage']['translated_copies'])
print(audit['leakage']['metadata_in_features'])

missing text: 0 | empty text: 0
Exact (lang,text) overlap between splits: {'train-val': 0, 'train-test': 0, 'val-test': 0}
ID overlap between splits: {'train-val': 0, 'train-test': 0, 'val-test': 0}
Near duplicates test->train (char 3-5gram TF-IDF cosine >= 0.85):
{
 "banking77": {
  "n_test": 1173,
  "n_near_dup": 30,
  "fraction": 0.02557544757033248,
  "threshold": 0.85,
  "median_max_similarity": 0.5003750405479714,
  "max_sim_p90": 0.7043215856297436,
  "near_dup_pairs_with_same_label": 29
 },
 "synthetic_enterprise": {
  "n_test": 34,
  "n_near_dup": 0,
  "fraction": 0.0,
  "threshold": 0.85,
  "median_max_similarity": 0.33478637064991856,
  "max_sim_p90": 0.475099814555667,
  "escalation_label_agreement": [
   0,
   0
  ],
  "urgency_label_agreement": [
   0,
   0
  ]
 }
}
Each Banking77 item appears in exactly one language (no id appears in two languages), so per-item translated copies across splits are not identifiable from this data; translations of the same original query MA

## 9. Code-mixing and the labels we do NOT have

In [10]:
cm = df[df.script == 'Deva+Latn']
print('Rows whose text mixes Devanagari + Latin letters:', len(cm), 'of', len(df))
print(cm.groupby('source_kind').size())
rom = df[(df.lang == 'hi-Latn')]
print('Romanised-Hindi (hi-Latn) rows:', len(rom), '| with intent label:', int(rom.intent.notna().sum()), '| with urgency label:', int(rom.urgency.notna().sum()))
print('NOT measurable with this data:'); [print(' -', x) for x in audit['not_measurable']]

Rows whose text mixes Devanagari + Latin letters: 39 of 8038
source_kind
Banking77 (intent)                           18
synthetic_enterprise (urgency/escalation)    21
dtype: int64
Romanised-Hindi (hi-Latn) rows: 135 | with intent label: 0 | with urgency label: 88
NOT measurable with this data:
 - Intent labels for Romanised Hindi / Hinglish: no such labelled data in the selected subsets
 - Intent+urgency+escalation on the same text: no text carries all labels


[None, None]

## 10. Class imbalance and the decision taken

In [11]:
print(json.dumps(audit['imbalance'], indent=1))
print('\nSanity baseline (non-neural) for synthetic escalation:'); print(json.dumps(audit['sanity_baseline_escalation_char_tfidf_lr_5fold_cv'], indent=1))

{
 "intent_train_classes": 77,
 "intent_min": 34,
 "intent_max": 105,
 "intent_max_over_min": 3.088235294117647,
 "urgency_train": {
  "1": 10,
  "2": 29,
  "3": 36,
  "4": 49
 },
 "escalation_train": {
  "1": 81,
  "0": 42
 },
 "decision": "No class weighting / resampling applied: intent classes are within ~2.6x of each other; urgency class 0 is rare (1 example overall) but cannot be rebalanced meaningfully."
}

Sanity baseline (non-neural) for synthetic escalation:
{
 "n": 177,
 "prevalence": 0.672316384180791,
 "pr_auc": 0.9547230298599954,
 "f1": 0.8166089965397924,
 "note": "Non-neural baseline; shows how learnable the synthetic escalation label is from surface text. CV folds are not grouped by near-duplicate templates, so this is optimistic."
}
